<a href="https://colab.research.google.com/github/ysl21292/Flyrank-MLE-intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ysl21292/Flyrank-MLE-intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


Task type: ranking (scoring).

Why: my question is "which pages should an editor check first?" The output is a list in order, with a score for each page. Under the hood, a model gives each page a chance of "views going down", and that chance becomes the score.

Action the output supports: an editor checks the top pages on the list and decides, for each one, to update it, protect it, or leave it alone.

One-paragraph frame: For a content editor deciding which pages to check first, I will build a ranked list from the starter data. Each page gets a score for its chance of "views going down", measured by Precision@50. A wrong call costs editor time on a fine page, or a missed page that keeps slipping. A plain rule is not enough because it only scores 17 of 30,000 pages. I will claim only observed and decision-support results.

In [1]:
import os, sys, subprocess
import pandas as pd, numpy as np

# In Colab: download the repo once. On your computer: go to the repo folder.
if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("Flyrank-MLE-intern"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/ysl21292/Flyrank-MLE-intern.git"], check=True)
    os.chdir("Flyrank-MLE-intern")
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} pages loaded")

30,000 pages loaded


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


Target: is the page's view count going down? (trend_direction == "down")

Where it comes from: it is measured in the data. It compares the last 30 days with the 30 days before. I did not invent it.

It is only a stand-in (a proxy) for what I really care about, which is pages that keep losing views in the future. Later I may switch to a future window (the next 30 days).

trend_direction and trend_pct are never used as inputs, because they are the answer in disguise.

In [2]:
df["is_declining"] = (df.trend_direction == "down").astype(int)
print(df.is_declining.value_counts().rename({1: "down", 0: "not down"}))
print(f"Share of pages going down: {df.is_declining.mean():.1%}")

is_declining
down        16262
not down    13738
Name: count, dtype: int64
Share of pages going down: 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Metric: Precision@50. Of the top 50 pages on my list, how many are really going down?

Why 50: I assume one editor can check about 50 pages. This is an assumption, not a measured number.

What "good" means: beating two things, on pages from clients the model has not seen.
1. Random guessing, which gets about 54%.
2. A simple hand-written rule.

I am choosing this metric now, before training any model.

In [3]:
def precision_at_k(scores, labels, k):
    top = np.argsort(-np.asarray(scores), kind="stable")[:k]
    return np.asarray(labels)[top].mean()

rule = ((df.days_since_last_update >= 180) & (df.impressions_90d >= 500)) * df.impressions_90d
print(f"Random guessing gets about {df.is_declining.mean():.2f}")
print(f"The simple rule scores only {int((rule > 0).sum())} pages.")
print(f"Simple rule, Precision@10: {precision_at_k(rule, df.is_declining, 10):.2f}")
print(f"Simple rule, Precision@17: {precision_at_k(rule, df.is_declining, 17):.2f}")

Random guessing gets about 0.54
The simple rule scores only 17 pages.
Simple rule, Precision@10: 1.00
Simple rule, Precision@17: 0.94


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page. The table below shows the first 5 pages. The last column is my target.

In [4]:
cols = ["impressions_90d", "clicks_90d", "avg_position", "days_since_last_update",
        "content_age_days", "trend_direction", "is_declining"]
print(df[cols].head(5).to_string())
print(f"One row = one page. {df.shape[0]:,} rows.")

   impressions_90d  clicks_90d  avg_position  days_since_last_update  content_age_days trend_direction  is_declining
0             3803          29          10.6                      20               187            down             1
1            15320           7          20.3                      25               445            down             1
2            12581          11          36.5                      20               141            down             1
3            11751          58           6.2                      22               463          stable             0
4            19140          24          44.0                      14               263            down             1
One row = one page. 30,000 rows.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


My simple rule ("not updated in 180+ days and 500+ views") is very good on the few pages it catches, but it only scores 17 pages out of 30,000. It has nothing to say about the rest.

The table below also shows that the pattern is mixed. At the top 3 positions, 4% of feedly articles are going down, but 37% of keyword articles are. The answer depends on several signals together, and one if-statement cannot rank all of that.

A model can weigh many signals at once and rank all 30,000 pages. But it only counts if it beats the simple rule on clients it has not seen. If it does not, the rule wins.

In [5]:
t = df.pivot_table(index="position_tier", columns="content_type",
                   values="is_declining", aggfunc=["mean", "size"])
rate = t["mean"].where(t["size"] >= 100).round(2)
print("Share of pages going down (blank = fewer than 100 pages):")
print(rate.to_string())

Share of pages going down (blank = fewer than 100 pages):
content_type   comparison article  feedly article  keyword article
position_tier                                                     
deep                          NaN             NaN             0.34
page_1                       0.57            0.47             0.58
page_3_5                      NaN             NaN             0.56
striking                     0.51            0.52             0.61
top_3                         NaN            0.04             0.37


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.